In [1]:
import sqlite3
import pandas as pd

# Pehle .db file ko git se ignore karo (badi file hai)
with open(".gitignore", "a") as f:
    f.write("*.db\n")

matches = pd.read_csv("data/matches_clean.csv")
full = pd.read_csv("data/ipl_clean.csv")

# Deliveries ke original 17 columns hi rakho (match wale extra columns hata do)
del_cols = ["match_id", "inning", "batting_team", "bowling_team", "over", "ball",
            "batter", "bowler", "non_striker", "batsman_runs", "extra_runs",
            "total_runs", "extras_type", "is_wicket", "player_dismissed",
            "dismissal_kind", "fielder"]
deliveries = full[del_cols]

conn = sqlite3.connect("data/ipl.db")
matches.to_sql("matches", conn, if_exists="replace", index=False)
deliveries.to_sql("deliveries", conn, if_exists="replace", index=False)

print(pd.read_sql("SELECT COUNT(*) AS matches FROM matches", conn))
print(pd.read_sql("SELECT COUNT(*) AS deliveries FROM deliveries", conn))

   matches
0     1095
   deliveries
0      260920


In [2]:
def q(sql):
    return pd.read_sql(sql, conn)


champions = q("""
    SELECT season, winner AS champion, date
    FROM matches
    WHERE match_type = 'Final'
    ORDER BY season
""")
print(champions)


titles = q("""
    SELECT winner AS team, COUNT(*) AS titles
    FROM matches
    WHERE match_type = 'Final'
    GROUP BY winner
    ORDER BY titles DESC
""")
print(titles)

    season               champion        date
0     2008       Rajasthan Royals  2008-06-01
1     2009        Deccan Chargers  2009-05-24
2     2010    Chennai Super Kings  2010-04-25
3     2011    Chennai Super Kings  2011-05-28
4     2012  Kolkata Knight Riders  2012-05-27
5     2013         Mumbai Indians  2013-05-26
6     2014  Kolkata Knight Riders  2014-06-01
7     2015         Mumbai Indians  2015-05-24
8     2016    Sunrisers Hyderabad  2016-05-29
9     2017         Mumbai Indians  2017-05-21
10    2018    Chennai Super Kings  2018-05-27
11    2019         Mumbai Indians  2019-05-12
12    2020         Mumbai Indians  2020-11-10
13    2021    Chennai Super Kings  2021-10-15
14    2022         Gujarat Titans  2022-05-29
15    2023    Chennai Super Kings  2023-05-29
16    2024  Kolkata Knight Riders  2024-05-26
                    team  titles
0         Mumbai Indians       5
1    Chennai Super Kings       5
2  Kolkata Knight Riders       3
3    Sunrisers Hyderabad       1
4      

In [3]:

potm = q("""
    SELECT player_of_match AS player, COUNT(*) AS awards
    FROM matches
    WHERE player_of_match != 'None'
    GROUP BY player_of_match
    ORDER BY awards DESC
    LIMIT 10
""")
print(potm)


orange = q("""
    WITH season_runs AS (
        SELECT m.season, d.batter, SUM(d.batsman_runs) AS runs
        FROM deliveries d
        JOIN matches m ON d.match_id = m.id
        GROUP BY m.season, d.batter
    ),
    ranked AS (
        SELECT season, batter, runs,
               RANK() OVER (PARTITION BY season ORDER BY runs DESC) AS rnk
        FROM season_runs
    )
    SELECT season, batter, runs
    FROM ranked
    WHERE rnk = 1
    ORDER BY season
""")
print(orange)

           player  awards
0  AB de Villiers      25
1        CH Gayle      22
2       RG Sharma      19
3         V Kohli      18
4       DA Warner      18
5        MS Dhoni      17
6       YK Pathan      16
7       SR Watson      16
8       RA Jadeja      16
9       SP Narine      15
    season         batter  runs
0     2008       SE Marsh   616
1     2009      ML Hayden   572
2     2010   SR Tendulkar   618
3     2011       CH Gayle   608
4     2012       CH Gayle   733
5     2013     MEK Hussey   733
6     2014     RV Uthappa   660
7     2015      DA Warner   562
8     2016        V Kohli   973
9     2017      DA Warner   641
10    2018  KS Williamson   735
11    2019      DA Warner   692
12    2020       KL Rahul   676
13    2021     RD Gaikwad   635
14    2022     JC Buttler   863
15    2023   Shubman Gill   890
16    2024        V Kohli   741


In [4]:
import os
os.makedirs("SQL", exist_ok=True)

queries = """-- IPL Analytics: SQL queries (SQLite)

-- 1. Har season ka champion
SELECT season, winner AS champion, date
FROM matches
WHERE match_type = 'Final'
ORDER BY season;

-- 2. Sabse zyada IPL titles
SELECT winner AS team, COUNT(*) AS titles
FROM matches
WHERE match_type = 'Final'
GROUP BY winner
ORDER BY titles DESC;

-- 3. Sabse zyada Player of the Match
SELECT player_of_match AS player, COUNT(*) AS awards
FROM matches
WHERE player_of_match != 'None'
GROUP BY player_of_match
ORDER BY awards DESC
LIMIT 10;

-- 4. Har season ka top run-scorer (window function + CTE)
WITH season_runs AS (
    SELECT m.season, d.batter, SUM(d.batsman_runs) AS runs
    FROM deliveries d
    JOIN matches m ON d.match_id = m.id
    GROUP BY m.season, d.batter
),
ranked AS (
    SELECT season, batter, runs,
           RANK() OVER (PARTITION BY season ORDER BY runs DESC) AS rnk
    FROM season_runs
)
SELECT season, batter, runs
FROM ranked
WHERE rnk = 1
ORDER BY season;
"""

with open("SQL/queries.sql", "w") as f:
    f.write(queries)
print("saved")

saved


In [5]:
deliveries.to_csv("data/deliveries_clean.csv", index=False)
print(deliveries.shape)

(260920, 17)


In [1]:
import pandas as pd

df = pd.read_csv("data/ipl_clean.csv")
df["season"] = df["season"].astype(int)

runs = df.groupby("season")["total_runs"].sum()
matches = df.groupby("season")["match_id"].nunique()
print((runs / matches).round(1))

season
2008    309.3
2009    286.9
2010    314.7
2011    289.8
2012    303.4
2013    297.4
2014    315.5
2015    311.1
2016    314.4
2017    318.4
2018    331.7
2019    323.9
2020    323.6
2021    310.6
2022    329.7
2023    347.1
2024    365.8
dtype: float64


In [2]:
readme = """# IPL Analytics (2008-2024)

Analysis of 17 IPL seasons using Python, SQL and Power BI.

**Data:** Kaggle IPL dataset, 1,095 matches and 260,920 deliveries.
**Tools:** Python (Pandas, Matplotlib, Seaborn), SQLite, Power BI, Git

## What I did
- Cleaned the data: unified team names, fixed season labels, merged duplicate venue names (58 to 36) and filled missing cities.
- Explored toss, venue, batting and bowling stats in Python.
- Wrote SQL queries (CTE and window functions) for champions, player of the match and top scorer per season.
- Built a 2-page Power BI dashboard with a season slicer.

## Findings
- Toss winners won 50.8% of matches, so the toss alone doesn't decide much. Teams that chose to field won 53.9% vs 45.4% for teams that chose to bat (this is correlation only).
- Chasing teams won 54.1% overall. Jaipur favours chasing (64.9%), Chennai favours defending (chase win rate 42.4%).
- Kohli has the most runs (8,014) and Chahal the most wickets (205).
- Mumbai and Chennai have 5 titles each.
- Average runs per match went from about 287 in 2009 to about 365 in 2024.

## Visuals
![Chase win % by venue](images/chase_by_venue.png)
![Dashboard overview](images/dashboard_overview.png)
![Dashboard players](images/dashboard_players.png)

Aniket | [GitHub](https://github.com/aniket78981)
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)
print("saved")

PermissionError: [Errno 13] Permission denied: 'README.md'

In [3]:
import os
print(os.path.isdir("README.md"))
print(os.listdir("README.md"))

True
[]


In [4]:
os.rmdir("README.md")
print(os.path.exists("README.md"))

False
